# Figure — R² Across Demographic Strata (Generalisable Model)

Grouped bar chart showing Train/Test R² for each demographic stratum:
Age (<40, 40-60, ≥60), BMI (<25, ≥25), Sex (Male, Female).

Data source: `DEMOG_RESULTS_PAPER / aggregated_{target}.csv`

Set `TARGET` below to switch between SBP / DBP / MAP.  
Design follows Fig 1/6: solid fill = Train, white-hatch overlay = Test.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path("../").resolve()))

from fig_style import (
    DPI, W_FULL, W_SINGLE, ASPECT, FONT_FAMILY,
    MIN_PX_FULL, MIN_PX_SINGLE,
    apply_base_style, save_fig,
)
from local_paths import DEMOG_RESULTS_PAPER, FIGURES_PAPER

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple

FIG_OUT = FIGURES_PAPER
FIG_OUT.mkdir(parents=True, exist_ok=True)
print(f"Output directory : {FIG_OUT.resolve()}")
print(f"Full-page  : {W_FULL:.2f} × {W_FULL*ASPECT:.2f} in  →  "
      f"{round(W_FULL*DPI)} × {round(W_FULL*ASPECT*DPI)} px @ {DPI} dpi")
print(f"Single-col : {W_SINGLE:.2f} × {W_SINGLE*ASPECT:.2f} in  →  "
      f"{round(W_SINGLE*DPI)} × {round(W_SINGLE*ASPECT*DPI)} px @ {DPI} dpi")

## 1 · Load data

In [ ]:
# ── Change TARGET here to produce DBP or MAP figures ─────────────────────────
TARGET = "SBP"

# Ordered strata: (segment key in CSV, display tick label, group label)
STRATA = [
    ("Age<40",    "<40",    "Age"),
    ("Age_40-60", "40-60",  "Age"),
    ("Age>=60",   "\u226560", "Age"),
    ("BMI<25",    "<25",    "BMI"),
    ("BMI>=25",   "\u226525", "BMI"),
    ("Gender=M",  "Male",   "Sex"),
    ("Gender=F",  "Female", "Sex"),
]
SEGMENTS   = [s[0] for s in STRATA]
TICK_LABELS = [s[1] for s in STRATA]
GROUP_LABELS = [s[2] for s in STRATA]

df_raw = pd.read_csv(DEMOG_RESULTS_PAPER / f"aggregated_{TARGET}.csv")
df_raw.columns = df_raw.columns.str.strip()

print("Datasets  :", sorted(df_raw["dataset"].unique()))
print("Segments  :", sorted(df_raw["segment"].unique()))
print("Variables :", sorted(df_raw["variable"].unique()))

## 2 · Plot

In [ ]:
# ── Palette ───────────────────────────────────────────────────────────────────
COLOR      = "#806D40"   # Sapphire blue — demographic strata figure
HATCH_TEST = "///"
BAR_WIDTH  = 0.35
LABEL_NUDGE = 0.09       # left nudge for Train, right nudge for Test

x = np.arange(len(STRATA))


def get_r2(df, target, segment, dataset):
    row = df[(df["target"] == target) &
             (df["segment"] == segment) &
             (df["dataset"] == dataset)]
    return float(row["R2"].values[0]) * 100 if len(row) else np.nan


def make_demog_fig(width_in: float, target: str = TARGET):
    _prev_hatch_lw = mpl.rcParams["hatch.linewidth"]
    mpl.rcParams["hatch.linewidth"] = 2.0

    height_in = width_in * ASPECT
    is_small  = width_in < 5
    label_fs  = 7   if is_small else 10
    tick_fs   = 8   if is_small else 11
    group_fs  = 9   if is_small else 12
    title_fs  = 8   if is_small else 12
    leg_fs    = 7   if is_small else 10

    train_vals = [get_r2(df_raw, target, seg, "train") for seg in SEGMENTS]
    test_vals  = [get_r2(df_raw, target, seg, "test")  for seg in SEGMENTS]

    fig, ax = plt.subplots(
        figsize=(width_in, height_in), dpi=DPI, layout="constrained"
    )

    all_bars = []
    for i in range(len(STRATA)):
        # Pass 1: solid fill + black edge
        bc_tr = ax.bar(i - BAR_WIDTH/2, train_vals[i], BAR_WIDTH,
                       color=COLOR, edgecolor="black", linewidth=0.8, zorder=3)
        bc_te = ax.bar(i + BAR_WIDTH/2, test_vals[i],  BAR_WIDTH,
                       color=COLOR, edgecolor="black", linewidth=0.8, zorder=3)
        # Pass 2: white-hatch overlay on Test bar
        ax.bar(i + BAR_WIDTH/2, test_vals[i], BAR_WIDTH,
               facecolor="none", hatch=HATCH_TEST,
               edgecolor="white", linewidth=0, zorder=3)

        all_bars.append((bc_tr[0], "train", train_vals[i]))
        all_bars.append((bc_te[0], "test",  test_vals[i]))

    # Data labels
    for bar, kind, val in all_bars:
        if not np.isnan(val):
            cx  = bar.get_x() + bar.get_width() / 2
            dx  = +LABEL_NUDGE if kind == "test" else 0
            ax.text(cx + dx, val + 0.4, f"{val:.2f}%",
                    ha="center", va="bottom",
                    fontsize=label_fs, fontfamily=FONT_FAMILY,
                    color="#333333")

    # Y-axis
    y_max = max(v for v in train_vals + test_vals if not np.isnan(v))
    ax.set_ylim(0, 100)   # headroom for labels
    ax.yaxis.set_major_locator(mticker.MultipleLocator(20))
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
    ax.set_ylabel("R²", fontsize=title_fs, fontfamily=FONT_FAMILY)

    # X-axis tick labels (stratum names)
    ax.set_xticks(x)
    ax.set_xticklabels(TICK_LABELS, fontsize=tick_fs, fontfamily=FONT_FAMILY)

    # Group labels (Age / BMI / Sex) below tick labels
    group_positions = {"Age": 1.0, "BMI": 3.5, "Sex": 5.5}
    for grp, cx in group_positions.items():
        ax.text(cx, -0.14, grp,
                ha="center", va="top",
                fontsize=group_fs, fontfamily=FONT_FAMILY,
                fontweight="bold",
                transform=ax.get_xaxis_transform())

    # Group dividers
    for xd in [2.5, 4.5]:
        ax.axvline(xd, color="#CCCCCC", linewidth=0.8, linestyle="--", zorder=0)

    apply_base_style(ax, grid_axis="y")
    ax.tick_params(axis="both", labelsize=tick_fs)

    # Title
    ax.set_title(
        f"R² Performance Across Demographic Strata — {target} (Generalisable Model)",
        fontsize=title_fs, fontfamily=FONT_FAMILY, pad=6,
    )

    # Legend
    test_handle = (
        Patch(facecolor=COLOR,     edgecolor="black", linewidth=0.8),
        Patch(facecolor="none",    edgecolor="white", hatch=HATCH_TEST, linewidth=0),
    )
    legend_elements = [
        Patch(facecolor=COLOR, edgecolor="black", linewidth=0.8, label="Train"),
        test_handle,
    ]
    ax.legend(
        handles=legend_elements,
        labels=["Train", "Test"],
        handler_map={tuple: HandlerTuple(ndivide=1, pad=0)},
        loc="upper right", ncol=1,
        fontsize=leg_fs,
        frameon=True, framealpha=0.9, edgecolor="#CCCCCC",
    )

    mpl.rcParams["hatch.linewidth"] = _prev_hatch_lw
    return fig


fig_full   = make_demog_fig(W_FULL)
fig_single = make_demog_fig(W_SINGLE)

save_fig(fig_full,   f"Fig_Demog_Strata_{TARGET}_full",   FIG_OUT)
save_fig(fig_single, f"Fig_Demog_Strata_{TARGET}_single", FIG_OUT)

print(f"Full-page  : {round(W_FULL*DPI)} × {round(W_FULL*ASPECT*DPI)} px")
print(f"Single-col : {round(W_SINGLE*DPI)} × {round(W_SINGLE*ASPECT*DPI)} px")
plt.show()

## 3 · Verify pixel counts

In [ ]:
from PIL import Image

for fname, req_w in [
    (f"Fig_Demog_Strata_{TARGET}_full.png",   MIN_PX_FULL),
    (f"Fig_Demog_Strata_{TARGET}_single.png", MIN_PX_SINGLE),
]:
    with Image.open(FIG_OUT / fname) as im:
        w, h = im.size
    ok = "\u2705" if w >= req_w else "\u274c"
    print(f"{ok} {fname}: {w} \u00d7 {h} px  (min required: {req_w} px wide)")